# Workspace Sync — Single Machine Example

This notebook demonstrates Monarch's workspace sync feature running entirely on one machine.

It uses `ProcessJob` to spawn local subprocesses that simulate remote hosts, then uses `rsync` to sync files from a "local workspace" to a "remote workspace" directory.

**Prerequisites:**
- `torchmonarch` installed (`pip install torchmonarch` or built from source)
- `rsync` installed (usually available by default on Linux)

## 1. Setup — Create local and remote workspace directories

In [ ]:
import shutil
import tempfile
from pathlib import Path

tmpdir = Path(tempfile.mkdtemp(prefix="sync_demo_"))
local_workspace = tmpdir / "local" / "my_project"
local_workspace.mkdir(parents=True)

remote_workspace_root = tmpdir / "remote" / "workspace"

print(f"Local workspace:  {local_workspace}")
print(f"Remote workspace: {remote_workspace_root}")

## 2. Create some project files locally

In [ ]:
(local_workspace / "train.py").write_text("print('training!')\n")
(local_workspace / "model.py").write_text("class MyModel: pass\n")

subdir = local_workspace / "configs"
subdir.mkdir()
(subdir / "config.yaml").write_text("lr: 0.001\nepochs: 10\n")

print("Local files:")
for f in sorted(local_workspace.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(local_workspace)}")

## 3. Create a Workspace config and start a ProcessJob

`Workspace` defines which local directories to sync. `ProcessJob` spawns local subprocesses that act as remote hosts.

In [ ]:
from monarch._src.job.process import ProcessJob
from monarch.tools.config.workspace import Workspace

workspace = Workspace(dirs=[local_workspace])

job = ProcessJob(
    {"hosts": 1},
    env={"WORKSPACE_DIR": str(remote_workspace_root)},
)
host = job.state(cached_path=None).hosts

print("ProcessJob started with 1 host")
print(f"Remote workspace root set to: {remote_workspace_root}")

## 4. Sync the workspace

This starts an rsync daemon locally, and the worker subprocess pulls files from it.

In [ ]:
await host.sync_workspace(workspace)
print("Sync complete!")

## 5. Verify — Check that files arrived on the remote side

In [ ]:
remote_project = remote_workspace_root / "my_project"

print("Remote files after sync:")
for f in sorted(remote_project.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(remote_project)} -> {f.read_text().strip()}")

## 6. Modify a file locally and re-sync

This demonstrates incremental sync — only changed files are transferred.

In [ ]:
(local_workspace / "train.py").write_text("print('training v2!')\n")
print("Modified train.py locally")

await host.sync_workspace(workspace)
print("Re-sync complete!")

print(f"\nRemote train.py now: {(remote_project / 'train.py').read_text().strip()}")

## 7. Add a new file and sync again

In [ ]:
(local_workspace / "utils.py").write_text("def helper(): return 42\n")
print("Added utils.py locally")

await host.sync_workspace(workspace)
print("Sync complete!")

print("\nAll remote files:")
for f in sorted(remote_project.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(remote_project)}")

## 8. Cleanup

In [ ]:
host.shutdown().get()
shutil.rmtree(tmpdir)
print("Cleaned up!")